Reference: https://www.kaggle.com/ryanholbrook/principal-component-analysis#kln-21

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.feature_selection import mutual_info_regression

from xgboost.sklearn import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import recall_score, f1_score, precision_score, precision_recall_fscore_support
from sklearn.metrics import accuracy_score

import seaborn as sns

In [2]:
plt.style.use("seaborn-whitegrid")
plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)


def plot_variance(pca, width=8, dpi=100):
    # Create figure
    fig, axs = plt.subplots(1, 2)
    n = pca.n_components_
    grid = np.arange(1, n + 1)
    # Explained variance
    evr = pca.explained_variance_ratio_
    axs[0].bar(grid, evr)
    axs[0].set(
        xlabel="Component", title="% Explained Variance", ylim=(0.0, 1.0)
    )
    # Cumulative Variance
    cv = np.cumsum(evr)
    axs[1].plot(np.r_[0, grid], np.r_[0, cv], "o-")
    axs[1].set(
        xlabel="Component", title="% Cumulative Variance", ylim=(0.0, 1.0)
    )
    # Set up figure
    fig.set(figwidth=8, dpi=100)
    return axs

def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_regression(X, y, discrete_features=discrete_features)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores


In [ ]:
theta_neg_w1 = ['c_w1_n1_theta', 'c_w1_n2_theta', 'c_w1_n3_theta']
theta_neg_w2 = ['c_w2_n1_theta', 'c_w2_n2_theta', 'c_w2_n3_theta']
theta_neg_w3 = ['c_w3_n1_theta', 'c_w3_n2_theta', 'c_w3_n3_theta']

theta_pos_w1 = ['c_w1_p1_theta', 'c_w1_p2_theta', 'c_w1_p3_theta']
theta_pos_w2 = ['c_w2_p1_theta', 'c_w2_p2_theta', 'c_w2_p3_theta']
theta_pos_w3 = ['c_w3_p1_theta', 'c_w3_p2_theta', 'c_w3_p3_theta']

deltas = ['n5s_delta', 'n15s_delta', 'n30s_delta', 'n60s_delta']


x_cols = ['c_w1_n1_theta', 'c_w1_n2_theta', 'c_w1_n3_theta',
          'c_w2_p1_theta', 'c_w2_p2_theta', 'c_w2_p3_theta', 
          'c_w3_n1_theta', 'c_w3_n2_theta', 'c_w3_n3_theta',
          'n5s_delta', 'n15s_delta', 'n30s_delta', 'n60s_delta']
y_cols = ['p30s_bucket']  # ,  'p5s_bucket', 'p30s_bucket', 'p60s_bucket']
bin_counts = 5

model_file_list = ['ml_cp18/ml_cp18_AAPL20220104.csv', 'ml_cp18/ml_cp18_AAPL20220106.csv']
test_file = "ml_cp18/ml_cp18_AAPL20220105-test.csv"


# Load Training Data

In [ ]:
data_tmp = pd.read_csv("../"+model_file_list[0], low_memory=False)
data_tmp.shape

In [ ]:
data = pd.read_csv("../"+model_file_list[1], low_memory=False)
data = data.append(data_tmp, ignore_index=True)
data.shape

In [ ]:
data = data.dropna()
data.shape

In [ ]:
x_train = data[ x_cols ]
y_train = data[ y_cols ].astype('int')

# Create XGB Model

In [ ]:
params = {
        'copy_X': True,
        'fit_intercept': False,
        'normalize': True,
        '_kfold': 5,
        # 'objective': 'validation:accuracy',
        'objective': 'multi:softprob',
        'enable_categorical': False,
        '_tuning_objective_metric': 'validation:f1',
        'eval_metric': 'auc',
        # 'tree_method': 'gpu_hist',
        # 'eval_metric': 'accuracy,f1',
        # 'eval_metric':'merror',
        # 'eval_metric': 'accuracy,f1X',
        # Muthu params
        'subsample': 0.75,  # Setting it to 0.5 means that XGBoost would randomly sample half of the training data prior to growing trees. and this will prevent overfitting.
        'gamma': 0.001, # Minimum loss reduction required to make a further partition on a leaf node of the tree. The larger gamma is, the more conservative the algorithm will be
        'alpha': 0.0, # L1 regularization is Lasso Regression wich adds “squared magnitude” of coefficient as penalty term to the loss functi
        'lambda': 1.0, # L2 regularization is Ridge Regression which adds “absolute value of magnitude” of coefficient as penalty term to the loss function.
        'eta': 0.03, # Step size shrinkage used in update to prevents overfitting
        'min_child_weight': 0.05, # Minimum sum of instance weight (hessian) needed in a child the building process will give up further partitioning.
        'num_class': bin_counts,
        # not too sure.
        'max_depth': 10,  # Maximum tree depth for base learners.
        'early_stopping_rounds': 50,  # don't over fit
        'num_round': 100,  # num_boost_round == num_boost_round
        'use_label_encoder': False # UserWarning: The use of label encoder in XGBClassifier is deprecated...
    }
xgb = XGBClassifier(**params)

In [ ]:
xgb.fit(x_train, y_train)

# Load Test Data

In [ ]:
test_data = pd.read_csv("../"+ test_file, low_memory=False)
test_data.shape

In [ ]:
all_cols = x_cols + y_cols
test_df = test_data [ all_cols ]
test_df = test_df.dropna()

x_test = test_df[ x_cols ]
y_test = test_df[ y_cols ].astype('int')

x_test.shape

# Base Prediction

In [ ]:
y_pred = xgb.predict(x_test)

In [ ]:
print('accuracy', accuracy_score( y_test[y_cols ], y_pred))

In [ ]:
precision, recall, f1, y_true = precision_recall_fscore_support( y_test [y_cols ], y_pred, average=None)
y_true = y_true.astype('int')
accuracy = accuracy_score(y_test[ y_cols ], y_pred)
print (f"accuracy: {accuracy:.3f}")

In [ ]:
pd.options.display.float_format = '{:,.3f}'.format
results_df =  pd.DataFrame( data = np.array ([recall,  precision, f1, y_true ] ),
                           columns = [ 'cat0', 'cat1', 'cat2', 'cat3','cat4'],
                           index = ['recall','precision', 'f1', 'y_true'] )
results_df

In [ ]:
sns.set(rc = {'figure.figsize':(12,9)})
sns.heatmap(confusion_matrix(y_test[ y_cols ], y_pred), annot=True, cmap="YlGnBu",fmt='d')

# Standard Scaled X Prediction

In [ ]:
scalar = StandardScaler()

scalar.fit(x_train)
x_scaled_train = scalar.transform(x_train)
x_scaled_test = scalar.transform(x_test)

In [ ]:
xgb_scaled = XGBClassifier(**params)
xgb_scaled.fit(x_scaled_train, y_train)

In [ ]:
y_scaled_pred = xgb_scaled.predict(x_scaled_test)

In [ ]:
precision, recall, f1, y_true = precision_recall_fscore_support( y_test[y_cols ], y_scaled_pred, average=None)
y_true = y_true.astype('int')
accuracy = accuracy_score(y_test[ y_cols ], y_scaled_pred)
print (f"accuracy: {accuracy:.3f}")

In [ ]:
pd.options.display.float_format = '{:,.3f}'.format
results_df =  pd.DataFrame( data = np.array ([recall,  precision, f1, y_true ] ),
                           columns = [ 'cat0', 'cat1', 'cat2', 'cat3','cat4'],
                           index = ['recall','precision', 'f1', 'y_true'] )
results_df

In [ ]:
sns.set(rc = {'figure.figsize':(12,9)})
sns.heatmap(confusion_matrix(y_test[ y_cols ], y_scaled_pred), annot=True, cmap="YlGnBu",fmt='d')

# Adding PCA

In [ ]:
from sklearn.decomposition import PCA
pca = PCA().fit(x_scaled_train)
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlim(0,7,1)
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')

In [ ]:
pca.explained_variance_ratio_

In [ ]:
plot_variance(pca)

In [ ]:
mi_scores = make_mi_scores(x_train, y_train[ y_cols ], discrete_features=False)
mi_scores